In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score, average_precision_score

from xgboost import XGBClassifier

# 데이터 로드
train_df = pd.read_csv(
    '../data/creditcard.csv'
)

print("중복 제거 전:", len(train_df))

train_df = train_df.drop_duplicates().reset_index(drop=True)

print("중복 제거 후:", len(train_df))

# 답 피처 구분
y_labels=train_df.iloc[:, -1].copy()
X_features= train_df.drop(columns=['Class']).copy()




중복 제거 전: 284807
중복 제거 후: 283726


In [3]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

roc_scores = []
ap_scores = []

In [5]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X_features, y_labels), start=1):

    print(f"\n===== Fold {fold} =====")

    X_train = X_features.iloc[train_idx].copy()
    X_val = X_features.iloc[val_idx].copy()

    y_train = y_labels.iloc[train_idx].copy()
    y_val = y_labels.iloc[val_idx].copy()

    # -------------------------
    # 1. Scaling
    # -------------------------
    scaler = RobustScaler()

    X_train['Amount_Scaled'] = scaler.fit_transform(
        X_train[['Amount']]
    )

    X_val['Amount_Scaled'] = scaler.transform(
        X_val[['Amount']]
    )

    X_train.drop(columns=['Time', 'Amount'], inplace=True)
    X_val.drop(columns=['Time', 'Amount'], inplace=True)

    # -------------------------
    # 2. Train 데이터 다시 결합
    # -------------------------
    train_df = X_train.copy()
    train_df['Class'] = y_train

    # -------------------------
    # 3. V14 이상치 제거
    # -------------------------
    fraud_v14 = train_df[
        train_df['Class'] == 1
    ]['V14']

    q25 = np.percentile(fraud_v14, 25)
    q75 = np.percentile(fraud_v14, 75)

    iqr = q75 - q25

    lower = q25 - 1.5 * iqr
    upper = q75 + 1.5 * iqr

    outlier_index = fraud_v14[
        (fraud_v14 < lower) |
        (fraud_v14 > upper)
    ].index

    train_df.drop(
        index=outlier_index,
        inplace=True
    )

    # -------------------------
    # 4. Fraud / Normal 분리
    # -------------------------
    fraud_train = train_df[
        train_df['Class'] == 1
    ].copy()

    normal_train = train_df[
        train_df['Class'] == 0
    ].copy()

    fraud_count = len(fraud_train)

    chunk_size = fraud_count * 10

    normal_train = normal_train.sample(
        frac=1,
        random_state=42 + fold
    ).reset_index(drop=True)

    # -------------------------
    # 5. 여러 10:1 샘플 생성
    # -------------------------
    sample_datasets = []

    for start in range(
        0,
        len(normal_train),
        chunk_size
    ):

        normal_chunk = normal_train.iloc[
            start:start + chunk_size
        ]

        if len(normal_chunk) < chunk_size:
            break

        sampled_df = pd.concat([
            fraud_train,
            normal_chunk
        ])

        sampled_df = sampled_df.sample(
            frac=1,
            random_state=42
        ).reset_index(drop=True)

        sample_datasets.append(sampled_df)

    print("샘플 수:", len(sample_datasets))

    # -------------------------
    # 6. XGB 여러 개 학습
    # -------------------------
    pred_proba_list = []

    for i, sampled_df in enumerate(sample_datasets):

        X_sample = sampled_df.drop(
            columns='Class'
        )

        y_sample = sampled_df['Class']

        model = XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='auc',
            random_state=42 + i,
            n_jobs=-1
        )

        model.fit(
            X_sample,
            y_sample
        )

        pred_proba = model.predict_proba(
            X_val
        )[:, 1]

        pred_proba_list.append(
            pred_proba
        )

    # -------------------------
    # 7. 앙상블 평균
    # -------------------------
    mean_pred_proba = np.mean(
        pred_proba_list,
        axis=0
    )

    # -------------------------
    # 8. Fold 평가
    # -------------------------
    roc = roc_auc_score(
        y_val,
        mean_pred_proba
    )

    ap = average_precision_score(
        y_val,
        mean_pred_proba
    )

    roc_scores.append(roc)
    ap_scores.append(ap)

    print("ROC-AUC:", roc)
    print("AP:", ap)


===== Fold 1 =====
샘플 수: 60
ROC-AUC: 0.9920066445614841
AP: 0.8408147880014236

===== Fold 2 =====
샘플 수: 60
ROC-AUC: 0.9837611549926633
AP: 0.8471740718669644

===== Fold 3 =====
샘플 수: 60
ROC-AUC: 0.9838390864257716
AP: 0.8264053676681519

===== Fold 4 =====
샘플 수: 61
ROC-AUC: 0.9763820318669578
AP: 0.8360361902252351

===== Fold 5 =====
샘플 수: 60
ROC-AUC: 0.9835663120732103
AP: 0.8499403137477392


In [6]:
print("\n==============================")
print("5-Fold 결과")
print("==============================")

print("ROC-AUC scores:")
print(roc_scores)

print(
    "ROC-AUC Mean:",
    np.mean(roc_scores)
)

print(
    "ROC-AUC Std:",
    np.std(roc_scores)
)

print("\nAP scores:")
print(ap_scores)

print(
    "AP Mean:",
    np.mean(ap_scores)
)

print(
    "AP Std:",
    np.std(ap_scores)
)


5-Fold 결과
ROC-AUC scores:
[0.9920066445614841, 0.9837611549926633, 0.9838390864257716, 0.9763820318669578, 0.9835663120732103]
ROC-AUC Mean: 0.9839110459840175
ROC-AUC Std: 0.004947145780324977

AP scores:
[0.8408147880014236, 0.8471740718669644, 0.8264053676681519, 0.8360361902252351, 0.8499403137477392]
AP Mean: 0.8400741463019028
AP Std: 0.008383786159502736
